In [1]:
from sklearn.metrics import log_loss, roc_auc_score
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt
from __future__ import annotations
from pathlib import Path
from typing import Union
import xgboost as xgb
from xgboost import XGBClassifier

In [2]:
'''dfs = [pd.read_parquet(f) for f in glob.glob("data/matches/*.parquet")]
match_df = dfs[np.random.randint(len(dfs))]'''

'dfs = [pd.read_parquet(f) for f in glob.glob("data/matches/*.parquet")]\nmatch_df = dfs[np.random.randint(len(dfs))]'

In [3]:
data = pd.read_parquet("data/processed/data_processed.parquet")

In [4]:
rand_id = (data['match_id'].unique())[np.random.randint(len(data['match_id'].unique()))]
#rand_id = '2023-usopen-1131'
match_df = data[data['match_id'] == rand_id]
(match_df)

,match_id,SetNo,GameNo,PointNumber,point_idx,elapsed_time,perspective,server_is_persp,pts_in_game_for,pts_in_game_against,...,ace_rate,avg_srv_speed_l60,fs_in_pct_l60,ss_in_pct_l60,fs_win_pct_l60,ss_win_pct_l60,ret_win_pct_l60,aces_l60,ace_rate_l60,y_match
2522586,2021-usopen-1408,1,1,500,1,False,False,False,0,0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0,0.000000,False
2522587,2021-usopen-1408,1,1,501,2,False,False,False,0,0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0,0.000000,False
2522588,2021-usopen-1408,1,1,502,3,False,False,False,0,0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0,0.000000,False
2522589,2021-usopen-1408,1,1,703,4,False,False,False,0,1,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0,0.000000,False
2522590,2021-usopen-1408,1,1,844,5,False,False,False,0,2,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0,0.000000,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2522895,2021-usopen-1408,3,8,626,153,False,True,True,3,0,...,0.168831,175.645161,0.0,0.0,0.0,0.0,0.413793,6,0.193548,True
2522896,2021-usopen-1408,3,9,628,154,False,True,False,4,0,...,0.166667,176.281250,0.0,0.0,0.0,0.0,0.392857,6,0.187500,True
2522897,2021-usopen-1408,3,9,630,155,False,True,False,1,0,...,0.166667,176.281250,0.0,0.0,0.0,0.0,0.392857,6,0.187500,True
2522898,2021-usopen-1408,3,9,632,156,False,True,False,2,0,...,0.166667,176.281250,0.0,0.0,0.0,0.0,0.392857,6,0.187500,True


In [5]:
def print_full(x):
    pd.set_option('display.max_rows', len(x))
    print(x)
    pd.reset_option('display.max_rows')

print_full(match_df)

                 match_id  SetNo  GameNo  PointNumber  point_idx  \
2522586  2021-usopen-1408      1       1          500          1   
2522587  2021-usopen-1408      1       1          501          2   
2522588  2021-usopen-1408      1       1          502          3   
2522589  2021-usopen-1408      1       1          703          4   
2522590  2021-usopen-1408      1       1          844          5   
2522591  2021-usopen-1408      1       1          969          6   
2522592  2021-usopen-1408      1       1         1085          7   
2522593  2021-usopen-1408      1       2         1168          8   
2522594  2021-usopen-1408      1       2         1182          9   
2522595  2021-usopen-1408      1       2         1200         10   
2522596  2021-usopen-1408      1       2         1223         11   
2522597  2021-usopen-1408      1       3          503         12   
2522598  2021-usopen-1408      1       3          529         13   
2522599  2021-usopen-1408      1       3        

In [8]:
match_df.to_csv("test.csv")

In [ ]:
def predict_win_probability(match_df, rand_id, model, feature_cols, names):
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    print("Log Loss:", log_loss(y_test, y_pred_prob))
    print("ROC AUC:", roc_auc_score(y_test, y_pred_prob))

    win_prob = []
    for _,point in match_df.iterrows():
        X_point = point[feature_cols].values.reshape(1, -1)
        prob = model.predict_proba(X_point)[0, 1]  # probability Player 1 wins match
        win_prob.append(prob)

    plt.plot(win_prob[:len(win_prob)//2], label='Win Probability (Player 1)')
    plt.xlabel('Point Index')
    plt.ylabel('Probability Player Wins Match')

    plt.plot(win_prob[len(win_prob)//2:], label='Win Probability (Player 2)')
    plt.title(f'Live Win Probability Prediction (Match ID: {rand_id}, Model: {names})')
    plt.ylim(0, 1)
    plt.legend()
    ax = plt.gca()
    draw_set_boundary_markers(ax, match_df)  # adds '--' for P1 and ':' for P2

    plt.show()



In [ ]:
model_df = pd.read_parquet("model_data.parquet", engine="pyarrow")
for i in range(len(model_df)):
    model = XGBClassifier()
    model.load_model(model_df['model_path'][i])
    feature_cols = model_df['input_columns'][i].split('$')
    predict_win_probability(match_df, rand_id, model, feature_cols, model_df['model_desc'][i])

In [ ]:
for i in match_df.columns:
    print((match_df[i]).head(1))


2684536    2021-wimbledon-2214
Name: match_id, dtype: object
2684536    1
Name: SetNo, dtype: int8
2684536    1
Name: GameNo, dtype: int8
2684536    500
Name: PointNumber, dtype: int16
2684536    1
Name: point_idx, dtype: int16
2684536    False
Name: elapsed_time, dtype: bool
2684536    False
Name: perspective, dtype: bool
2684536    False
Name: server_is_persp, dtype: bool
2684536    0
Name: pts_in_game_for, dtype: int8
2684536    0
Name: pts_in_game_against, dtype: int8
2684536    0
Name: games_in_set_for, dtype: int8
2684536    0
Name: games_in_set_against, dtype: int8
2684536    0
Name: sets_for, dtype: int8
2684536    0
Name: sets_against, dtype: int8
2684536    5
Name: best_of, dtype: int8
2684536    3
Name: sets_needed_to_win, dtype: int8
2684536    False
Name: is_tiebreak, dtype: bool
2684536    False
Name: is_game_point_for, dtype: bool
2684536    False
Name: is_game_point_against, dtype: bool
2684536    False
Name: is_break_point, dtype: bool
2684536    0
Name: ttl_diff, dtyp

In [ ]:
(match_df['ss_in_pct_l60']).head(1)

2684536    0.0
Name: ss_in_pct_l60, dtype: float64